In [4]:
from dataclasses import dataclass

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest, after_model

load_dotenv()


@dataclass
class Context:
    user_name: str


# Dynamic prompts
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}."
    return system_prompt


from langchain.agents import AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime


# Before model hook
@before_model
def log_before_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Processing request for user: {runtime.context.user_name}")
    return None


# After model hook
@after_model
def log_after_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Completed request for user: {runtime.context.user_name}")
    return None


agent = create_agent(
    model="deepseek-flash",
    tools=[],
    middleware=[dynamic_system_prompt, log_before_model, log_after_model],
    context_schema=Context
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")
)

for m in response['messages']:
    m.pretty_print()

Processing request for user: John Smith
Completed request for user: John Smith
================================ Human Message =================================

What's my name?
================================== Ai Message ==================================

Your name is John Smith.
